In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

In [ ]:
df = pd.read_csv("../Hotel Reservations.csv")
df = df.drop(columns = ["Booking_ID"])

x = df.drop(columns = ["booking_status"])
y = df["booking_status"]

le = LabelEncoder()
y = le.fit_transform(y)

numeric_features = x.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = x.select_dtypes(include = ['object']).columns

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [ ]:
numeric_pipeline = Pipeline(steps = [("imputer", SimpleImputer(strategy = "median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline(steps = [("imputer", SimpleImputer(strategy = "most_frequent")), ("onehot", OneHotEncoder(handle_unknown = "ignore"))])
preprocessor = ColumnTransformer(transformers = [("num", numeric_pipeline, numeric_features), ("cat", categorical_pipeline, categorical_features)])

In [ ]:
#################
# RANDOM FOREST #
#################

baseline_model = Pipeline(steps = [('preprocessor', preprocessor), ('model', RandomForestClassifier(random_state = 42))])
baseline_model.fit(x_train, y_train)

y_test_pred_baseline = baseline_model.predict_proba(x_test)[:, 1]
baseline_test_roc_auc = roc_auc_score(y_test, y_test_pred_baseline)

print(f"Baseline Test ROC AUC: {baseline_test_roc_auc:.4f}")

Baseline Test ROC AUC: 0.9569


In [ ]:
#########################
# HYPERPARAMETER TUNING #
#########################

param_grid = {"model__n_estimators": [50, 100, 200],
              "model__max_depth": [None, 10, 20],
              "model__min_samples_split": [2, 5],
              "model__min_samples_leaf": [1, 2]}

grid_search = GridSearchCV(estimator = baseline_model,
                           param_grid = param_grid,
                           scoring = "roc_auc",
                           cv = 3,
                           n_jobs = -1,
                           verbose = 1)

print("\nStarting Grid Search...")
grid_search.fit(x_train, y_train)

print("\nBest Params:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)


Starting Grid Search...
Fitting 3 folds for each of 36 candidates, totalling 108 fits


c:\Users\adria\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Best Params: {'model__max_depth': 20, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best CV Score: 0.9492856150756784


In [ ]:
#######################
# FINAL EVALUATION    #
#######################

best_model = grid_search.best_estimator_
y_test_pred_best = best_model.predict_proba(x_test)[:, 1]
best_test_roc_auc = roc_auc_score(y_test, y_test_pred_best)

print(f"Baseline Test ROC AUC:   {baseline_test_roc_auc:.4f}")
print(f"Best Model Test ROC AUC: {best_test_roc_auc:.4f}")

y_pred_best_class = best_model.predict(x_test)
print("\nClassification Report (Best Model):")
print(classification_report(y_test, y_pred_best_class))

------------------------------
Baseline Test ROC AUC:   0.9572
Best Model Test ROC AUC: 0.9581
------------------------------

Classification Report (Best Model):
              precision    recall  f1-score   support

           0       0.89      0.81      0.85      2377
           1       0.91      0.95      0.93      4878

    accuracy                           0.91      7255
   macro avg       0.90      0.88      0.89      7255
weighted avg       0.91      0.91      0.90      7255



In [ ]:
mlp_pipeline = Pipeline(steps = [('preprocessor', preprocessor), ('model', MLPClassifier(hidden_layer_sizes = (64, 32),
                                                                                         max_iter = 100, 
                                                                                         early_stopping = True, 
                                                                                         random_state = 42))])

print("Training Baseline MLP...")
mlp_pipeline.fit(x_train, y_train)

y_val_pred_mlp = mlp_pipeline.predict_proba(x_test)[:, 1]
baseline_mlp_auc = roc_auc_score(y_test, y_val_pred_mlp)
print(f"Baseline MLP ROC AUC: {baseline_mlp_auc:.4f}")

param_grid_mlp = {"model__hidden_layer_sizes": [(50,), (100,), (50, 25)], "model__alpha": [0.0001, 0.001], "model__activation": ["relu", "tanh"]}

grid_search_mlp = GridSearchCV(estimator = mlp_pipeline, param_grid = param_grid_mlp, scoring = "roc_auc", cv = 3, n_jobs = -1, verbose = 1)

print("\nStarting MLP Grid Search...")
grid_search_mlp.fit(x_train, y_train)

print(f"Best MLP Params: {grid_search_mlp.best_params_}")

best_mlp = grid_search_mlp.best_estimator_
y_pred_best_mlp = best_mlp.predict_proba(x_test)[:, 1]
best_mlp_auc = roc_auc_score(y_test, y_pred_best_mlp)

print(f"Baseline MLP AUC: {baseline_mlp_auc:.4f}")
print(f"Tuned MLP AUC:    {best_mlp_auc:.4f}")

Training Baseline MLP...
Baseline MLP ROC AUC: 0.9289

Starting MLP Grid Search...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best MLP Params: {'model__activation': 'tanh', 'model__alpha': 0.001, 'model__hidden_layer_sizes': (50, 25)}
------------------------------
Baseline MLP AUC: 0.9289
Tuned MLP AUC:    0.9249
------------------------------
